## 2.5 文本预处理 - Padding、Truncation定长化与 batch 文本如何送入 RNN

#### 1、为什么这一节必须学习

##### 1.1 前面我们已经解决了“单个 token 如何变成向量”的问题
到上一节为止，我们已经知道了这样一条链路：

原始文本  
$\rightarrow$ 分词  
$\rightarrow$ token 序列  
$\rightarrow$ token 对应 id  
$\rightarrow$ Embedding  
$\rightarrow$ 每个 token 变成一个向量

例如一句话：

`I love AI`

最后可以变成一个向量序列：

$x_1, x_2, x_3$

也就是说，从“单个句子”的角度看，文本已经可以送进 RNN 了。

但是这里还有一个非常现实的问题没有解决：

不同句子的长度通常不一样。

并且加入 batch 并不只是单一维度地增加词或字的数量，而是**新增一个 batch 维度**。  
这就会直接影响训练时的 batch 组织。

##### 1.2 真正训练模型时，不是一次只输入一句话，而是一次输入一个 batch
在实际训练中，我们通常不会每次只喂给模型一条样本，因为那样效率太低。

更常见的方式是：

一次输入一个 batch

例如：

- 16 条句子
- 32 条句子
- 64 条句子

但是一旦把多个句子放到一起，就会发现一个问题：

有的句子短，  
有的句子长。

例如：

`I love AI`  
长度是 `3`

`This movie is very good`  
长度是 `5`

`Deep learning is changing the world`  
长度是 `6`

那么问题来了：

这些长度不同的句子，怎么放进同一个张量里？

因为加入 batch 并不只是简单地把所有 token 叠加起来，而是新增一个 batch 维度。

比如：

- `I love AI`，长度是 `3`
- `This movie is very good`，长度是 `5`
- `Deep learning is changing the world`，长度是 `6`

这里的 `batch_size = 3`，并不是简单地叠加 token 数量，而是表示：

同一批次里有 `3` 条样本。

##### 1.3 Padding 和 Truncation 定长化是文本 batch 化的关键步骤
如果不做统一长度处理，那么句子长度不一致，模型就没法把它们拼成规则张量。

而神经网络，尤其是 batch 训练，最喜欢规则的张量结构。

所以这一节的核心任务就是理解：

- 为什么句子要定长化
- 什么是 padding
- 什么是 truncation
- batch 文本最后是怎样组织成张量的
- padding 后为什么才能送进 RNN


#### 2、为什么不同长度的文本不能直接组成 batch

##### 2.1 每个句子的 token 数量天然不一样
自然语言不像固定长度的表格特征，它的长度往往是变化的。

例如下面 3 个句子：

`I love AI`  
长度是 `3`

`This movie is very good`  
长度是 `5`

`Deep learning is changing the world`  
长度是 `6`

即使它们都已经转成 id 序列，也仍然是不同长度：

```python
[2, 3, 4]
[8, 9, 5, 6, 7]
[10, 11, 12, 13]
```

##### 2.2 张量要求形状规则
神经网络在 batch 训练时，通常希望输入是规则张量。

例如：

$3 \times 5$

表示：

- `3` 条样本
- 每条样本长度都是 `5`

但刚才那 3 条句子的长度分别是：

`3`、`5`、`4`

由于增加 batch 时，是直接增加一个新维度，而不是直接叠加 token 数量，  
也就是说，你不能直接写成：

```python
[
    [2, 3, 4],
    [8, 9, 5, 6, 7],
    [10, 11, 12, 13]
]
```

因为每一行长度不同，这样的结构不是规则张量。

##### 2.3 所以必须先统一长度
为了让一个 batch 能够组成张量，必须先让所有句子的长度一致。

常见做法就是：

- 太短的句子补齐
- 太长的句子截断

这样所有样本就会拥有统一长度，最终才能构造成：

$batch\_size \times seq\_len$

的 id 张量，再经过 Embedding 变成：

$batch\_size \times seq\_len \times embedding\_dim$


#### 3、什么是定长化

##### 3.1 定长化就是把所有句子处理成同一个长度
所谓定长化，可以理解为：

把原本长短不一的序列，整理成统一长度的序列。

它通常包括两种操作：

- Padding：短的补齐
- Truncation：长的截断

也就是说，定长化不是只做 padding，而是：

短句补，长句裁，最终都变成固定长度。

##### 3.2 定长化是“工程需要”，不一定是语言本身需要
从语言本身来说，句子当然可以长短不同。

但从神经网络批量训练的角度看，为了计算方便，我们通常要把它们整理成相同长度。

所以定长化更多是一个模型输入层面的工程处理步骤。

#### 4、什么是 Padding

##### 4.1 Padding 的本质
Padding 的意思就是：

对较短的句子进行补齐，让它们达到统一长度。

这个“补齐”的内容通常不是普通单词，而是一个特殊 token：

`<PAD>`

它的作用只是占位，不表示真实语义。

##### 4.2 一个最简单的例子
假设我们规定：

所有句子统一长度为 `5`

那么：

`I love AI`

原始 id 序列可能是：

```python
[2, 3, 4]
```

长度只有 `3`，所以要补齐到 `5`：

```python
[2, 3, 4, 0, 0]
```

其中 `0` 可能就是 `<PAD>` 的 id。

##### 4.3 Padding 的作用不是增加语义，而是让形状统一
这一点非常重要。

Padding 不是为了让句子更完整，也不是为了补充信息。

它只是为了把不同长度的句子，强行整理成统一长度，便于 batch 训练。

所以你可以把 `<PAD>` 理解成：

“空白占位符”

它的核心作用只有一个：

统一形状。

#### 5、什么是 Truncation

##### 5.1 Truncation 的本质
Truncation 的意思是：

如果句子太长，就把多余部分截掉。

例如我们规定：

统一长度为 `5`

某个句子对应的 id 序列是：

```python
[8, 9, 10, 11, 12, 13, 14]
```

长度是 `7`，超过了 `5`。

那么就可以截断成：

```python
[8, 9, 10, 11, 12]
```

这就叫 truncation。

##### 5.2 为什么需要截断
因为如果只做 padding，不做截断，那么超长句子仍然没法统一进固定长度的 batch 中。

例如目标长度是 `5`，但某个句子长度是 `20`，那它还是没法直接塞进统一长度的 batch。

所以定长化一定是双向处理：

- 短句补齐
- 长句截断

##### 5.3 截断会带来信息损失
这一点也很重要。

Padding 只是加占位，不改变原句有效信息。

但 truncation 会真正删掉部分内容。

所以当你决定统一长度时，实际上是在做一个平衡：

- 长度太小，会丢掉更多信息
- 长度太大，会增加计算成本和无效 padding

因此统一长度的选择，通常要结合数据实际情况来决定。


#### 6、Padding 和 Truncation 一起工作是什么样的

##### 6.1 看一个统一长度为 5 的例子
假设有 3 个句子，它们已经转成 id 序列：

句子 A：

```python
[2, 3, 4]
```

长度 `3`

句子 B：

```python
[5, 6, 7, 8, 9]
```

长度 `5`

句子 C：

```python
[10, 11, 12, 13, 14, 15]
```

长度 `6`

现在规定统一长度为 `5`。

##### 6.2 对短句做 padding
句子 A：

```python
[2, 3, 4]
```

$\rightarrow$

```python
[2, 3, 4, 0, 0]
```

##### 6.3 对刚好够长的句子不处理
句子 B：

```python
[5, 6, 7, 8, 9]
```

$\rightarrow$

```python
[5, 6, 7, 8, 9]
```

##### 6.4 对长句做 truncation
句子 C：

```python
[10, 11, 12, 13, 14, 15]
```

$\rightarrow$

```python
[10, 11, 12, 13, 14]
```

##### 6.5 最终 batch 可以组织成规则张量
```python
[
    [2, 3, 4, 0, 0],
    [5, 6, 7, 8, 9],
    [10, 11, 12, 13, 14]
]
```

现在它就是一个形状规则的二维张量：

$3 \times 5$

也就是：

- `batch_size = 3`
- `seq_len = 5`

#### 7、定长化后 batch 文本如何进入 RNN

##### 7.1 第一步：先得到统一长度的 id 张量
例如：

```python
[
    [2, 3, 4, 0, 0],
    [5, 6, 7, 8, 9],
    [10, 11, 12, 13, 14]
]
```

形状是：

$batch\_size \times seq\_len = 3 \times 5$

##### 7.2 第二步：经过 Embedding
假设：

$embedding\_dim = 4$

那么这个 $3 \times 5$ 的 id 张量，经过 Embedding 后会变成：

$3 \times 5 \times 4$

含义是：

- `3` 条句子
- 每条句子 `5` 个 token
- 每个 token 对应一个 `4` 维向量

##### 7.3 第三步：输入 RNN
这时，RNN 接收的输入就是：

$batch\_size \times seq\_len \times embedding\_dim$

也就是：

$3 \times 5 \times 4$

RNN 会对每条句子的每个时间步依次处理。

所以从整体流程来看：

原始文本  
$\rightarrow$ 分词  
$\rightarrow$ id 序列  
$\rightarrow$ padding / truncation  
$\rightarrow$ 统一长度的 batch id 张量  
$\rightarrow$ Embedding  
$\rightarrow$ RNN

这就是 batch 文本进入 RNN 的完整主流程。


#### 8、Padding token 在 Embedding 里怎么处理

##### 8.1 `<PAD>` 也会有自己的 id
通常我们会给 `<PAD>` 一个固定 id，例如：

`<PAD> → 0`

那么在补齐时，所有空位都用 `0` 来填充。

例如：

```python
[2, 3, 4]
```

$\rightarrow$

```python
[2, 3, 4, 0, 0]
```

##### 8.2 `<PAD>` 理论上也会经过 Embedding
因为整个序列都会送入 Embedding 层，所以 `<PAD>` 的 id 也会被映射成一个向量。

例如：

```python
0 -> [0.0, 0.0, 0.0, 0.0]
```

当然，这只是一个理想化例子。

##### 8.3 实际中通常希望 PAD 尽量不带语义干扰
因为 `<PAD>` 不是真实词，它只是占位符。

所以我们通常不希望它像普通单词一样携带强语义信息。

在很多框架里，会专门处理 padding 位置，让它尽量不影响模型学习。

例如在 PyTorch 中，Embedding 可以设置 `padding_idx`。

这样模型会知道：

这个 id 是专门用来 padding 的，需要特殊对待。


#### 9、为什么 padding 会带来新问题

##### 9.1 RNN 会不会把 PAD 也当成真实输入处理
这是 padding 最大的问题之一。

例如一个句子本来真实长度只有 `3`：

```python
[2, 3, 4]
```

但补齐后变成：

```python
[2, 3, 4, 0, 0]
```

如果直接送进 RNN，那么第 `4`、`5` 个时间步其实也是会被处理的。

这就意味着：

RNN 并不知道哪些是“真实 token”，哪些只是“补出来的 PAD”。

##### 9.2 这样可能会引入无效计算
PAD 本身不携带真实语义，但 RNN 仍然会对这些位置做计算，更新隐藏状态。

这会导致：

- 额外的无效计算
- 可能干扰真实序列表示
- 长短句混合时影响更明显

##### 9.3 所以后面通常还要进一步处理“变长序列问题”
这一节我们先建立对 padding 的整体理解。

但你要提前有一个意识：

Padding 解决了 batch 形状统一问题，  
却又引入了“PAD 会不会干扰模型”的新问题。

这就是为什么后面我们还要学习：

- `padding_idx`
- `mask`
- `pack_padded_sequence`

这些方法本质上都是在进一步解决：

如何让模型知道哪些位置是真实数据，哪些位置只是 padding。